In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config

In [0]:
display(v_data_source)
display(v_file_date)

In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
circuits_schema = StructType(fields=[StructField("circuitId", StringType(), False),
                                     StructField("circuitRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("location", StringType(), True),
                                     StructField("country", StringType(), True),
                                     StructField("lat", StringType(), True),
                                     StructField("lng", StringType(), True),
                                     StructField("alt", StringType(), True),
                                     StructField("url", StringType(), True)
])

# COMMAND ----------

circuits_df = spark.read \
.option("header", True) \
.schema(circuits_schema) \
.csv(f"{raw_folder_path}/{v_file_date}/circuits.csv")

In [0]:
from pyspark.sql.functions import col

circuits_selected_df = circuits_df.select(col("circuitId"), col("circuitRef"), col("name"), col("location"), col("country"), col("lat"), col("lng"), col("alt"))

from pyspark.sql.functions import lit

circuits_renamed_df = (circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") 
.withColumnRenamed("circuitRef", "circuit_ref") 
.withColumnRenamed("lat", "latitude") 
.withColumnRenamed("lng", "longitude") 
.withColumnRenamed("alt", "altitude") 
)

circuits_final_df = add_ingestion_date(circuits_renamed_df)
circuits_final_df= (circuits_final_df
                    .withColumn("data_source", lit(v_data_source)) 
                    .withColumn("file_date", lit(v_file_date)))
                    
circuits_final_df.write.mode("append").format("delta").option("mergeSchema","true").saveAsTable("f1.bronze.circuits")
dbutils.notebook.exit("Success")